# Phase 01: Leakage-Safe Dataset Preparation, Duplicate-Aware Stratified Splitting, EDA & Frozen Manifests
## Framework: XAI-RiceGuard
**Project Title:** *A Lesion-Grounded Explainable and Uncertainty-Aware Deep Learning Framework for Robust Rice Leaf Blast and Brown Spot Detection*

---

> **RESEARCH INTEGRITY & HARD GATE BOUNDARY:**
> - **Phase 01 Objective:** Establish an immutable, leakage-free dataset partition (Train 70%, Validation 10%, Calibration 10%, Internal Test 10%) at the atomic duplicate-group level, profile image resolution statistics, generate 8 publication-grade EDA figures, and freeze cryptographic manifest checksums.
> - **Strict Boundary:** NO model training, NO data augmentation for training, NO XAI heatmap generation, and NO temperature scaling calibration fitting are performed in this phase.
> - **Immutable Separation:** Sethy 5932 and BD5 3150 are 100% held out as external benchmarks and never enter the primary development splits.

### Step 1: Environment & System Verification
Detect operating system, Python runtime, CPU/GPU accelerator status, and package dependencies.

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.phase0.environment import detect_environment, generate_environment_report

env_info = detect_environment()
print(generate_environment_report(env_info))

### Step 2: Load Central Configuration and Resolve Paths
Mount Google Drive (if on Colab) and resolve dataset roots and Phase 1 artifact directories.

In [ ]:
if env_info['is_colab']:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")

from src.phase0.config import resolve_paths, load_project_config

paths = resolve_paths()
proj_config = load_project_config()
seed = proj_config.get("reproducibility", {}).get("default_seed", 42)

print(f"Active Environment: {paths['environment'].upper()}")
print(f"Project Root      : {paths['project_root']}")
print(f"Reproducibility Seed: {seed}")

### Step 3: Load Phase 0 Manifests and Validate Completeness
Load verified manifests for RiceLeafDiseaseBD (17,963), Sethy (5,932), BD5 (3,150), and verify exact count matching.

In [ ]:
from src.phase1.manifest_loader import load_phase0_manifests, validate_manifest_completeness

manifests = load_phase0_manifests(manifests_dir=paths["artifacts"]["manifests"])
df_primary = manifests["primary"]
df_sethy = manifests["sethy"]
df_bd5 = manifests["bd5"]

comp_res = validate_manifest_completeness(manifests)
print("=" * 60)
print("Manifest Completeness Audit:")
for k, v in comp_res["details"].items():
    print(f"  - {k:<12}: Actual={v['actual']:,} | Expected={v['expected']:,} -> {'PASS' if v['passed'] else 'FAIL'}")
print(f"Overall Status: {'PASS' if comp_res['all_passed'] else 'FAIL'}")

### Step 4: Primary Dataset Disease Label Analysis
Analyze class distributions, sample counts, percentages, and calculate the overall class imbalance ratio.

In [ ]:
from src.phase1.label_analysis import analyze_primary_labels, export_class_distributions

manifests_p1_dir = Path(paths["project_root"]) / "manifests" / "phase1"
df_dist, label_summary = analyze_primary_labels(df_primary)
export_class_distributions(df_dist, label_summary, output_dir=str(manifests_p1_dir))

print("=" * 60)
print("RiceLeafDiseaseBD: Disease Class Distribution")
print("=" * 60)
display(df_dist)
print(f"\nTotal Samples  : {label_summary['total_samples']:,}")
print(f"Imbalance Ratio: {label_summary['imbalance_ratio']}:1 (Max: {label_summary['largest_class']}, Min: {label_summary['smallest_class']})")

### Step 5: Deterministic Duplicate Group Construction
Group images sharing exact SHA-256 hashes into atomic duplicate clusters to prevent data leakage.

In [ ]:
from src.phase1.duplicate_groups import build_duplicate_groups

df_primary_grouped, df_groups, dup_summary = build_duplicate_groups(df_primary)

print("=" * 60)
print("Duplicate Hash Grouping Summary")
print("=" * 60)
print(f"Total Images                     : {dup_summary['total_images']:,}")
print(f"Unique SHA-256 Hashes            : {dup_summary['total_unique_hashes']:,}")
print(f"Duplicate Groups (>1 image)      : {dup_summary['total_duplicate_groups']:,}")
print(f"Unique Groups (1 image)          : {dup_summary['total_unique_groups']:,}")
print(f"Total Images in Duplicate Groups : {dup_summary['total_images_in_duplicate_groups']:,}")
print(f"Max Duplicate Cluster Size       : {dup_summary['max_images_in_single_group']}")

### Step 6: Group-Aware Stratified 4-Way Splitting (Train / Val / Cal / Test)
Execute class-by-class proportional group allocation with seed 42 to partition images into Train (70%), Validation (10%), Calibration (10%), and Internal Test (10%) while strictly preserving $G_i \cap G_j = \emptyset$.

In [ ]:
from src.phase1.split_generator import generate_group_aware_stratified_splits

df_splits, split_dfs, split_summary = generate_group_aware_stratified_splits(
    df_primary_grouped, seed=seed
)

print("=" * 60)
print("Group-Aware Stratified Split Proportions")
print("=" * 60)
for s_name, s_info in split_summary["splits"].items():
    print(f"  - {s_name.replace('_', ' ').title():<14}: {s_info['sample_count']:>5,} samples ({s_info['actual_percentage']:>5.2f}%) [Target: {s_info['target_percentage']}%]")

### Step 7: Split Integrity & Leakage Firewall Validation
Formally audit that zero duplicate groups cross split boundaries, primary and external datasets have zero SHA-256 collisions, and external datasets are firewalled.

In [ ]:
from src.phase1.split_validator import validate_phase1_splits

val_res = validate_phase1_splits(df_splits, split_dfs, df_sethy, df_bd5)

print("=" * 60)
print("Phase 1 Hard-Gate Validation Checks")
print("=" * 60)
for chk, info in val_res["checks"].items():
    status = "PASS" if info["passed"] else "FAIL"
    print(f"  [{status}] {chk.replace('_', ' ').title():<28}: {info['detail']}")

print(f"\nOverall Validation Passed: {val_res['overall_passed']}")

### Step 8: Image Resolution & Dimension Statistics
Compute resolution metrics (mean/median width/height, aspect ratios, file sizes).

In [ ]:
from src.phase1.dataset_statistics import compute_image_dimension_statistics

dim_stats = compute_image_dimension_statistics(df_splits)
print("=" * 60)
print("Primary Dataset Image Dimension Summary")
print("=" * 60)
for k, v in dim_stats["dimensions"].items():
    print(f"  - {k:<14}: Mean={v['mean']}, Median={v['median']}, Min={v['min']}, Max={v['max']}")
print(f"Top Resolutions: {dim_stats['top_resolutions']}")

### Step 9: Freeze Manifests and Compute Cryptographic SHA-256 Checksums
Export all Phase 1 manifests to `manifests/phase1/` and generate `manifest_checksums.json`.

In [ ]:
from src.phase1.split_generator import freeze_manifests

freeze_res = freeze_manifests(
    df_all_splits=df_splits,
    split_dfs=split_dfs,
    df_duplicate_groups=df_groups,
    df_sethy=df_sethy,
    df_bd5=df_bd5,
    output_dir=str(manifests_p1_dir)
)

# Write empty exclusion log (0 exclusions required as all images are verified readable)
exclusion_df = pd.DataFrame(columns=["sample_id", "reason", "source", "action"])
exclusion_df.to_csv(manifests_p1_dir / "exclusion_log.csv", index=False)

print("=" * 60)
print("Cryptographic Manifest Checksums (Frozen Experimental Baseline)")
print("=" * 60)
for fname, sha in freeze_res["checksums"].items():
    print(f"  - {fname:<30}: {sha}")

### Step 10: Generate 8 Publication-Grade EDA Figures
Render and export figures to `figures/phase1/` and save the deterministic sample registry `manifests/phase1/eda_samples.csv`.

In [ ]:
from src.phase1.eda import generate_all_phase1_figures

figures_p1_dir = Path(paths["project_root"]) / "figures" / "phase1"
fig_paths, df_samples = generate_all_phase1_figures(
    df_primary_splits=df_splits,
    split_dfs=split_dfs,
    df_sethy=df_sethy,
    df_bd5=df_bd5,
    dataset_roots=paths["dataset_roots"],
    output_dir=str(figures_p1_dir),
    seed=seed
)
df_samples.to_csv(manifests_p1_dir / "eda_samples.csv", index=False)

print("=" * 60)
print("Generated Publication-Grade EDA Figures:")
print("=" * 60)
for name, fpath in fig_paths.items():
    print(f"  - {name:<28}: {fpath}")

### Step 11: Compile Markdown Reports and Phase 01 Completion Checklist
Generate `phase1_dataset_statistics.md`, `phase1_split_report.md`, `phase1_validation_report.md`, `phase1_completion_report.md`, and `phase1_run_metadata.json`.

In [ ]:
from src.phase1.phase1_report import generate_all_phase1_reports

reports_p1_dir = Path(paths["project_root"]) / "reports" / "phase1"
report_paths = generate_all_phase1_reports(
    env_info=env_info,
    label_summary=label_summary,
    duplicate_summary=dup_summary,
    split_summary=split_summary,
    validation_results=val_res,
    dim_stats_primary=dim_stats,
    manifest_checksums=freeze_res["checksums"],
    output_dir=str(reports_p1_dir)
)

print("=" * 60)
print("Generated Phase 1 Reports & Metadata:")
print("=" * 60)
for name, rpath in report_paths.items():
    print(f"  - {name:<22}: {rpath}")

with open(report_paths["completion_report"], "r", encoding="utf-8") as f:
    print("\n" + f.read())